In [ ]:
!pip install websockets
!pip install yfinance

In [ ]:

#!pip install requests.html

In [ ]:
import yfinance as yf
import pandas as pd
import time

In [ ]:
#ticker = yf.Ticker("AAPL")
#balance_sheet = ticker.balancesheet
#print(balance_sheet)

In [ ]:
balance_sheet = []
income_statement = []
cfs = []
years = []
profitability_score = 0
leverage_score = 0
operating_efficiency_score = 0
pe_ratio = 0

summary = pd.DataFrame(columns = ['Ticker', 'Year', 'ROE',
                                  'ROA', 'ROFL',
                                  'Profit Margin', 'Asset Turnover',
                                 'APT', 'ART', 'INVT', 'PPET', 'C2C', 'S&GA/Revenue'])

# The original line 'tickers = yf.tickers_sp500()' caused an HTTP Error 403: Forbidden.
# This error indicates that the website being accessed by yahoo-fin (likely Wikipedia)
# for the S&P 500 ticker list has blocked the automated request.
# As a workaround, we will use a small hardcoded list of tickers to allow the rest of the code to run.
# For a comprehensive list, consider an alternative data source or a dedicated API.
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN'] # Example hardcoded list

In [ ]:
def get_data(ticker):
    global balance_sheet
    global income_statement
    global cfs
    global years
    tickeryf =  yf.Ticker(ticker)
# Get the balance sheet
    balance_sheet = tickeryf.balance_sheet
   # print(balance_sheet)
# Get the income statement (referred to as 'financials' in the library)
    income_statement = tickeryf.income_stmt
    # print(income_statement)
    cfs = tickeryf.cash_flow
    #print(cfs)
    years = balance_sheet.columns
   # print(ticker, years)

In [ ]:
def get_calcs():

    global ROA, ROE, ROFL, Profit_Margin, Asset_Turnover, APT, ART, INVT, PPET, C2C, SGR
    global ROA_py, ROE_py, ROFL_py, Profit_Margin_py, Asset_Turnover_py, APT_py, ART_py, INVT_py, PPET_py, C2C_py, SGR_py

    netIncome = income_statement[years[0]]['Net Income From Continuing Operation Net Minority Interest']
    netIncome_py = income_statement[years[1]]['Net Income From Continuing Operation Net Minority Interest']

    totalAssets = balance_sheet[years[0]]['Total Assets'] # Corrected key
    totalAssets_py = balance_sheet[years[1]]['Total Assets'] # Corrected key

    interestExpense = income_statement[years[0]]['Interest Expense'] # Corrected key for yfinance
    interestExpense_py = income_statement[years[1]]['Interest Expense'] # Corrected key for yfinance

    #print("Balance Sheet Index (Financial Metrics):")
    #print(balance_sheet.index.tolist())
    # The key 'Total Stockholder Equity' is causing KeyError.
    # We will calculate Total Shareholder Equity from Total Assets and Total Liabilities.
    # Assuming 'Total Liabilities' is the correct key for total liabilities.
    totalLiabilities = balance_sheet[years[0]]['Total Liabilities Net Minority Interest']
    totalLiabilities_py = balance_sheet[years[1]]['Total Liabilities Net Minority Interest']

    totalShareholderEquity = totalAssets - totalLiabilities
    totalShareholderEquity_py = totalAssets_py - totalLiabilities_py

    totalRevenue = income_statement[years[0]]['Total Revenue'] # Assuming this key is still correct or closest
    totalRevenue_py = income_statement[years[1]]['Total Revenue'] # Assuming this key is still correct or closest

    cogs = income_statement[years[0]]['Total Revenue'] - income_statement[years[0]]['Gross Profit'] # These might need adjustment too
    cogs_py = income_statement[years[1]]['Total Revenue'] - income_statement[years[1]]['Gross Profit'] # These might need adjustment too

    netReceivables = balance_sheet[years[0]]['Accounts Receivable'] # Corrected key
    netReceivables_py = balance_sheet[years[1]]['Accounts Receivable'] # Corrected key

    inventory = balance_sheet[years[0]]['Inventory'] # Assuming this key is still correct or closest
    inventory_py = balance_sheet[years[1]]['Inventory'] # Assuming this key is still correct or closest

    pPE  = balance_sheet[years[0]]['Net PPE'] # Corrected key for propertyPlantEquipment
    pPE_py  = balance_sheet[years[1]]['Net PPE'] # Corrected key for propertyPlantEquipment

    #print("Income Statement Index (Financial Metrics):")
    #print(income_statement.index.tolist())
    sga = income_statement[years[0]]['Selling General And Administration'] # Assuming this key is still correct or closest
    sga_py = income_statement[years[1]]['Selling General And Administration'] # Assuming this key is still correct or closest

    accountsPayable = balance_sheet[years[0]]['Accounts Payable'] # Assuming this key is still correct or closest
    accountsPayable_py = balance_sheet[years[1]]['Accounts Payable'] # Assuming this key is still correct or closest


    ROE = netIncome/totalShareholderEquity

    ROA = (netIncome + (1-0.35)*interestExpense )/totalAssets

    ROFL = ROE - ROA


    Profit_Margin = (netIncome + (1-0.35)*interestExpense )/totalRevenue

    Asset_Turnover = totalRevenue/totalAssets

    APT = cogs/accountsPayable

    ART = totalRevenue/netReceivables

    INVT = cogs/inventory

    PPET = totalRevenue/pPE

    C2C = -1.0/APT + 1.0/ART + 1.0/INVT

    SGR = sga/totalRevenue

    ROE_py = netIncome_py/totalShareholderEquity_py


    ROA_py = (netIncome_py + (1-0.35)*interestExpense_py )/totalAssets_py

    ROFL_py = ROE_py - ROA_py

    Profit_Margin_py = (netIncome_py + (1-0.35)*interestExpense_py )/totalRevenue_py

    Asset_Turnover_py = totalRevenue_py/totalAssets_py

    APT_py = cogs_py/accountsPayable_py

    ART_py = totalRevenue_py/netReceivables_py

    INVT_py = cogs_py/inventory_py

    PPET_py = totalRevenue_py/pPE_py

    C2C_py = -1.0/APT_py + 1.0/ART_py + 1.0/INVT_py

    SGR_py = sga_py/totalRevenue_py

I added a cell below to show how to get data if given a ticker symbol.

In [ ]:
ticker = 'AAPL'
#get_data(ticker)
#print("did THAT")

get_data(ticker)
get_calcs()

new_row = {'Ticker': ticker,'Year': years[0],
                   'ROE': ROE,
                   'ROA': ROA, 'ROFL': ROFL,
                   'Profit Margin': Profit_Margin, 'Asset Turnover': Asset_Turnover,
                   'APT':APT, 'ART':ART, 'INVT':INVT, 'PPET':PPET, 'C2C':C2C, 'S&GA/Revenue':SGR}

summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index = True)
new_row = {'Ticker': ticker,'Year': years[1],
                   'ROE': ROE_py,
                   'ROA': ROA_py, 'ROFL': ROFL_py,
                   'Profit Margin': Profit_Margin_py, 'Asset Turnover': Asset_Turnover_py,
                   'APT':APT_py, 'ART':ART_py, 'INVT':INVT_py, 'PPET':PPET_py, 'C2C':C2C_py, 'S&GA/Revenue':SGR_py}

summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index = True)
print(ticker + ' added.')
time.sleep(3)

summary.to_csv('Summary.csv')
summary

Here you can loop through multiple symbols

In [ ]:
for ticker in tickers[0:4]:
    try:
        get_data(ticker)
        get_calcs()

        new_row = {'Ticker': ticker,'Year': years[0],
                   'ROE': ROE,
                   'ROA': ROA, 'ROFL': ROFL,
                   'Profit Margin': Profit_Margin, 'Asset Turnover': Asset_Turnover,
                   'APT':APT, 'ART':ART, 'INVT':INVT, 'PPET':PPET, 'C2C':C2C, 'S&GA/Revenue':SGR}

        summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index = True)
        new_row = {'Ticker': ticker,'Year': years[1],
                   'ROE': ROE_py,
                   'ROA': ROA_py, 'ROFL': ROFL_py,
                   'Profit Margin': Profit_Margin_py, 'Asset Turnover': Asset_Turnover_py,
                   'APT':APT_py, 'ART':ART_py, 'INVT':INVT_py, 'PPET':PPET_py, 'C2C':C2C_py, 'S&GA/Revenue':SGR_py}

        summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index = True)
        print(ticker + ' added.')
        time.sleep(3)
    except:
        print(ticker + ': Something went wrong.')

summary.to_csv('Summary.csv')

In [ ]:
summary